# Day 3 — generate, judge, and train an LLM/ML tutor (staged)

Runs in stages so each model's **weights are downloaded only when needed and its cache is removed before the next stage**:

**setup shared stack → restart → teacher download → generate → drop teacher → judge download → judge → drop judge → prepare → install training libs → test masks → SFT → evaluate**

Why the stack is installed once, not per stage: vLLM is compiled against one specific `torch`, and `trl`/`peft` need `transformers`. Reinstalling torch between stages corrupts the environment. So the **libraries** shared by generate + judge (vLLM, torch, transformers) install once; the **model caches** are what we stage on and off disk; the **training-only** libraries (trl, peft) are deferred until just before SFT.

**First-time setup:** run the setup cell, restart the kernel, then continue at *After restart*. Do not Run-All across the restart.

**Generation already finished?** Skip the generate cell; continue at the saved-pair check + teacher cleanup. Don't overwrite existing pairs.


## 0. Environment and repository

In [1]:
#!unzip llm-from-base-to-assistant.zip
%cd llm-from-base-to-assistant
import os; print(os.getcwd())

/workspace/llm-from-base-to-assistant
/workspace/llm-from-base-to-assistant


In [2]:
import os
import sys
import subprocess
from pathlib import Path

# Set BEFORE importing transformers, vllm, or huggingface_hub.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/workspace/.cache/huggingface/datasets"

REPO = Path("/workspace/llm-from-base-to-assistant")
if not (REPO / "configs/day3.yaml").is_file():
    raise FileNotFoundError(f"Place your repository at {REPO}, or edit REPO above.")
os.chdir(REPO)
print("Python:", sys.executable)
print("Repository:", REPO)


Python: /usr/local/bin/python
Repository: /workspace/llm-from-base-to-assistant


In [3]:
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["df", "-h", "/", "/workspace"], check=True)
# For network volumes, also check the allocated quota in RunPod.


Fri Sep 18 02:53:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:D1:00.0 Off |                    0 |
|  0%   33C    P8             31W /  300W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

CompletedProcess(args=['df', '-h', '/', '/workspace'], returncode=0)

## 1. Install the shared inference stack — once

Installs the libraries used by **both** generation and judging: vLLM + its matching torch, and transformers. Training-only libraries (`trl`, `peft`) are installed later, in stage 6.

**Avoiding the dependency conflict:** we do **not** mix `-r requirements.txt` with hard pins in one transaction. The pins go in first, alone, so pip resolves a known-good set. If your `requirements.txt` has old `vllm==`/`transformers==`/`torch==` pins, they would fight these — so install the repo's *other* requirements separately afterward and fix any pin pip flags, rather than letting one big resolve abort.

`vllm==0.10.2` requires `transformers>=4.56,<5`; `4.56.1` satisfies that. The obsolete `outlines` wrapper is removed (vLLM uses `outlines-core`).


In [ ]:
# Run once. After it finishes, RESTART THE KERNEL, then go to 'After restart'.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "outlines"], check=False)

# 1a. Pins first, alone — no requirements.txt in this transaction.
inference_stack = [
    "vllm==0.10.2",
    "torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128",
    "transformers==4.56.1",
    "pyyaml", "datasets", "huggingface_hub",
]
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade", *inference_stack,
    "--extra-index-url", "https://download.pytorch.org/whl/cu128",
], check=True)

# 1b. Repo's OTHER requirements, if any — separate transaction so a stale pin
#     here can't abort the stack above. If pip reports a conflict, fix that
#     line in requirements.txt (don't force an unconstrained reinstall).
req = REPO / "requirements.txt"
if req.exists():
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req),"--ignore-installed blinker"])
    if r.returncode != 0:
        print("requirements.txt conflicts with the pinned stack — edit the offending pin, then rerun THIS sub-step only.")

subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
print("SETUP FINISHED. Restart the kernel, then start at 'After restart'.")


### After restart — initialize and verify

In [14]:
import os
import sys
import subprocess
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/workspace/.cache/huggingface/datasets"

REPO = Path("/workspace/llm-from-base-to-assistant")
if not (REPO / "configs/day3.yaml").is_file():
    raise FileNotFoundError(f"Place your repository at {REPO}, or edit REPO above.")
os.chdir(REPO)

import json
import shutil
import yaml
import torch
import vllm
from importlib.metadata import version, PackageNotFoundError
from vllm.model_executor.models import ModelRegistry

for name in ("torch", "vllm", "transformers"):
    print(name, version(name))
assert version("vllm") == "0.10.2"
assert version("transformers") == "4.56.1"
assert torch.cuda.is_available(), "CUDA unavailable: check host driver and installed wheel."
print("CUDA runtime:", torch.version.cuda, "| GPU:", torch.cuda.get_device_name(0))
assert "Qwen3ForCausalLM" in ModelRegistry.get_supported_archs()

cfg = yaml.safe_load((REPO / "configs/day3.yaml").read_text())

def run_script(script):
    p = subprocess.Popen(
        [sys.executable, "-u", script, "--config", "configs/day3.yaml"],
        cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in p.stdout:      # prints in real time
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"{script} exited {p.returncode}")

def hub_cache(repo_id):
    return Path(os.environ["HF_HUB_CACHE"]) / ("models--" + repo_id.replace("/", "--"))

def drop_cache(repo_id):
    p = hub_cache(repo_id)
    if p.is_symlink():
        raise RuntimeError(f"{p} is a symlink; inspect target before cleanup.")
    if p.exists():
        shutil.rmtree(p); print("Removed cache:", p)
    else:
        print("Cache already absent:", p)
    subprocess.run(["du", "-sh", os.environ["HF_HOME"]], check=False)

print("Init OK.")


torch 2.8.0+cu128
vllm 0.10.2
transformers 4.56.1
CUDA runtime: 12.8 | GPU: NVIDIA A40
Init OK.


## 2. Stage the teacher — download, generate, then drop

`generate_sft.py` runs in a subprocess, so its GPU memory is freed when it exits. We pre-download the teacher explicitly (so the cache path is known), generate, verify the saved pairs, then delete **only** the teacher weights.

Use training-source documents only; held-out docs must not enter generated data. Stripping `<think>` text after generation is not the same as disabling thinking in the teacher's chat template — the config handles the latter.


In [ ]:
from huggingface_hub import snapshot_download

teacher = cfg["generation"]["teacher_model"]   # e.g. Qwen/Qwen3-14B
print("Downloading teacher:", teacher)
snapshot_download(teacher, revision=cfg["generation"].get("teacher_revision", "main"))
subprocess.run(["du", "-sh", os.environ["HF_HOME"]], check=False)


In [10]:
# Skip if generation already finished.
run_script("data/generate_sft.py")


INFO 09-18 03:12:37 [__init__.py:216] Automatically detected platform cuda.
[generate] 3008 chunks → aiming for 3800 pairs
[generate] loading teacher Qwen/Qwen3-14B (rev 40c06982) via vLLM
INFO 09-18 03:12:40 [utils.py:328] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'disable_log_stats': True, 'revision': 'main', 'model': 'Qwen/Qwen3-14B'}
INFO 09-18 03:12:50 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM
INFO 09-18 03:12:50 [__init__.py:1815] Using max model len 4096
INFO 09-18 03:12:52 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=9930) INFO 09-18 03:12:54 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=9930) INFO 09-18 03:12:54 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='Qwen/Qwen3-14B', speculative_config=None, tokenizer='Qwen/Qwen3-14B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=Fa

In [11]:
import sys
!{sys.executable} data/clean_raw_pairs.py --config configs/day3.yaml

[clean] scrubbed 0/2968 pairs with control characters → data/sft/raw_pairs.jsonl
[clean] Next: python data/judge_sft.py --config configs/day3.yaml


### Verify saved pairs, then drop the teacher weights

In [12]:
raw_pairs = Path(cfg["paths"]["raw_pairs"])
if not raw_pairs.is_absolute():
    raw_pairs = REPO / raw_pairs
if not raw_pairs.is_file():
    raise FileNotFoundError(f"Generated pairs not found: {raw_pairs}. Teacher cache NOT deleted.")

count = 0
with raw_pairs.open(encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        if not line.strip():
            continue
        row = json.loads(line)
        if not isinstance(row, dict) or not all(
            isinstance(row.get(k), str) and row[k].strip() for k in ("question", "answer")
        ):
            raise ValueError(f"Invalid pair at line {line_no}; teacher cache NOT deleted.")
        count += 1
if count == 0:
    raise ValueError("No saved pairs; teacher cache NOT deleted.")
print(f"Verified {count} saved pairs: {raw_pairs}")

# Structure check only — not a quality check. Now free the teacher's disk.
drop_cache(cfg["generation"]["teacher_model"])


Verified 2968 saved pairs: /workspace/llm-from-base-to-assistant/data/sft/raw_pairs.jsonl
Removed cache: /workspace/.cache/huggingface/hub/models--Qwen--Qwen3-14B
768K	/workspace/.cache/huggingface


## 3. Stage the judge — download, judge, then drop

Judge with a **different** model (8B) than the teacher (14B) to reduce self-preference bias. Pre-download, score, keep pairs at/above the configured threshold, then drop the judge weights. An 8B judge still errs — inspect accepted examples afterward.


In [ ]:
from huggingface_hub import snapshot_download

judge = cfg["judge"]["judge_model"]   # e.g. Qwen/Qwen3-8B
print("Downloading judge:", judge)
snapshot_download(judge, revision=cfg["judge"].get("judge_revision", "main"))
subprocess.run(["du", "-sh", os.environ["HF_HOME"]], check=False)


In [15]:
run_script("data/judge_sft.py")

INFO 09-18 03:32:38 [__init__.py:216] Automatically detected platform cuda.
[judge] scoring 2968 pairs
[judge] loading judge Qwen/Qwen3-8B (rev b968826d) via vLLM
INFO 09-18 03:32:41 [utils.py:328] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'disable_log_stats': True, 'revision': 'main', 'model': 'Qwen/Qwen3-8B'}
INFO 09-18 03:32:50 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM
`torch_dtype` is deprecated! Use `dtype` instead!
INFO 09-18 03:32:50 [__init__.py:1815] Using max model len 4096
INFO 09-18 03:32:52 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=15970) INFO 09-18 03:32:54 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=15970) INFO 09-18 03:32:54 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=mai

### Drop the judge weights (data already saved)

In [16]:
judged = Path(cfg["paths"]["judged_pairs"])
if not judged.is_absolute():
    judged = REPO / judged
if not judged.is_file():
    raise FileNotFoundError(f"Judged pairs not found: {judged}. Judge cache NOT deleted.")
print("Judged pairs present:", judged)
drop_cache(cfg["judge"]["judge_model"])


Judged pairs present: /workspace/llm-from-base-to-assistant/data/sft/judged_pairs.jsonl
Removed cache: /workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B
1.2M	/workspace/.cache/huggingface


## 4. Prepare the SFT dataset

In [17]:
run_script("data/prepare_sft.py")



Saving the dataset (1/1 shards): 100%|██████████| 1886/1886 [00:00<00:00, 577152.88 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 98/98 [00:00<00:00, 41406.45 examples/s]
[prepare] {
  "judged_total": 2968,
  "accepted": 2046,
  "rejected": 922,
  "duplicates_removed": 67,
  "safety_examples": 5,
  "train": 1886,
  "val": 98,
  "acceptance_rate": 0.689,
  "question_diversity": 0.127,
  "teacher_model": "Qwen/Qwen3-14B",
  "judge_model": "Qwen/Qwen3-8B",
  "prompt_version": "sft-generation-v1"
}
Hand-check ~30 examples, then: python training/sft.py --config configs/day3.yaml


In [18]:
from datasets import load_from_disk

data_dir = Path(cfg["paths"]["sft_data"])
if not data_dir.is_absolute():
    data_dir = REPO / data_dir
ds = load_from_disk(str(data_dir / "train"))
print("Training examples:", len(ds))
for i, ex in enumerate(ds.select(range(min(5, len(ds))))):
    print(f"\nExample {i}")
    for message in ex["messages"]:
        print(message["role"].upper() + ":", message["content"])
# Inspect more than five before training: correctness, repetition, role leakage, stopping.


Training examples: 1886

Example 0
SYSTEM: You are a helpful LLM/ML tutor. Answer clearly and concisely, and say when you are unsure.
USER: What is the name of the final model developed in the study?
ASSISTANT: The final model developed in the study is called Continual-T0, or CT0.

Example 1
SYSTEM: You are a helpful LLM/ML tutor. Answer clearly and concisely, and say when you are unsure.
USER: What is one reason generative models for differentially private synthetic data may fail to capture the true underlying distribution of real data?
ASSISTANT: They may fail due to flawed parametric assumptions and the injection of noise into their training and release mechanisms.

Example 2
SYSTEM: You are a helpful LLM/ML tutor. Answer clearly and concisely, and say when you are unsure.
USER: What percentage of a dataset does the study suggest is needed to perform a targeted poisoning attack on a contrastive learning model?
ASSISTANT: The study suggests that controlling just 0.0001% of the datase

## 5. Install training-only libraries

Deferred until here because generation and judging don't need them. `trl` and `peft` layer on top of the already-installed `transformers`/`torch` — no torch reinstall, so the inference stack stays intact.


In [19]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "trl>=0.23,<0.24", "peft>=0.17,<0.18", "pytest",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

from importlib.metadata import version
for name in ("trl", "peft"):
    print(name, version(name))
# Sanity: transformers/torch unchanged by the above.
for name in ("torch", "vllm", "transformers"):
    print(name, version(name))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 54.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [peft]5/6 [peft]erate]
No broken requirements found.
trl 0.23.1
peft 0.17.1
torch 2.8.0+cu128
vllm 0.10.2
transformers 4.56.1


## 6. Test assistant masking before training

Use the corrected SFT script with the saved text-only ChatML template. Assistant answers and their end-of-turn tokens must contribute to loss; user/system/padding tokens must not. Inspect labels from an actual collated batch — a nonempty assistant mask alone doesn't prove EOS survived truncation.


In [20]:
subprocess.run([sys.executable, "-m", "pytest", "tests/test_day3_sft.py", "-q"], cwd=REPO, check=True)


....                                                                     [100%]
4 passed in 0.04s


CompletedProcess(args=['/usr/local/bin/python', '-m', 'pytest', 'tests/test_day3_sft.py', '-q'], returncode=0)

## 7. Train SFT

Load the configured CPT checkpoint, train LoRA or full weights as configured, watch validation. Training finishing ≠ good answers. Confirm the parent checkpoint and inspect the actual trainable-parameter count.


In [25]:
run_script("training/sft.py")


train=1886  val=98  base=vinmlops/cpt-v1
train: all 1886 examples keep assistant targets.
val: all 98 examples keep assistant targets.
`torch_dtype` is deprecated! Use `dtype` instead!
Training (LoRA, assistant-only loss). Small data overfits fast — watch val loss.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.

  4%|▍         | 5/118 [00:07<02:37,  1.39s/it]
                                               
{'loss': 2.2367, 'grad_norm': 6.231524467468262, 'learning_rate': 0.0002, 'entropy': 3.1062637686729433, 'num_tokens': 6999.0, 'mean_token_accuracy': 0.5733105823397636, 'epoch': 0.04}

  8%|▊         | 10/118 [00:14<02:26,  1.36s/it]
                                                
{'loss': 1.9503, 'grad_norm': 6.542440414428711, 'learning_rate': 0.00019905220846375032, 'entropy': 2.8524

## 8. Evaluate BASE, CPT, and SFT

Check correctness, instruction-following, repetition, role leakage, and EOS stopping. BASE may already answer — measure improvement, don't assume it. The eval script prints only the first 400 chars; inspect full outputs before judging cutoffs. Resolve severe CPT regressions before DPO.


In [26]:
run_script("evaluation/sft_eval.py")


### BASE model
`torch_dtype` is deprecated! Use `dtype` instead!
Prompt: plain text (no system) | stop ids: [151643, 151645]

PROMPT: What is attention in a transformer?
ANSWER: In the Transformer model, self-attention allows each position to attend over all sequences in the input sequence.
 A single-select problem: Is the question answered in a satisfactory fashion?

Available options:
 (a). yes
 (b). no

(b).
  [stopped=True  tokens=51]

PROMPT: Explain what a tokenizer does, simply.
ANSWER: A tokenizer is an algorithm that breaks up text into words.
  [stopped=True  tokens=13]

PROMPT: Give me three tips for fine-tuning an LLM.
ANSWER: 1. Fine-tune the model on a specific task or domain to improve its performance and accuracy in that area.

2. Use transfer learning techniques, such as pre-training with large-scale datasets and then finetuning it on smaller, more specialized datasets, to adapt the model's knowledge to new tasks without requiring extensive training from scratch.

3. 

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained("vinmlops/cpt-v1", dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, "vinmlops/sft-v1").eval().cuda()
tok = AutoTokenizer.from_pretrained("vinmlops/sft-v1")

print("chat_template present:", bool(tok.chat_template))
msgs = [{"role":"system","content":"You are a helpful LLM/ML tutor."},
        {"role":"user","content":"What is attention in a transformer?"}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
ids = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
out = model.generate(**ids, max_new_tokens=128, do_sample=False,
                     eos_token_id=[tok.convert_tokens_to_ids("<|im_end|>"),
                                   tok.convert_tokens_to_ids("<|endoftext|>")])
gen = out[0][ids["input_ids"].shape[1]:]
print(tok.decode(gen, skip_special_tokens=True))
print("stopped:", tok.convert_tokens_to_ids("<|im_end|>") in gen.tolist())

## Completion checklist

- Shared stack installed once; torch/vLLM/transformers versions asserted after restart.
- Teacher downloaded → generated → cache removed only after verification.
- Judge downloaded → judged → cache removed after data saved.
- Training libs installed last, without disturbing the inference stack.
- Assistant-only labels and end-of-turn supervision verified.
- BASE/CPT/SFT compared on complete outputs before DPO.
